In [5]:
import sys
!{sys.executable} -m pip install openmeteo-requests requests-cache retry-requests pandas numpy

  Using cached openmeteo_requests-1.7.5-py3-none-any.whl.metadata (11 kB)
  Using cached requests_cache-1.3.1-py3-none-any.whl.metadata (9.4 kB)
  Using cached retry_requests-2.0.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached niquests-3.18.2-py3-none-any.whl.metadata (15 kB)
  Using cached openmeteo_sdk-1.25.0-py3-none-any.whl.metadata (935 bytes)
  Using cached cattrs-26.1.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached url_normalize-2.2.1-py3-none-any.whl.metadata (5.6 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached urllib3_future-2.17.902-py3-none-any.whl.metadata (16 kB)
  Using cached wassima-2.0.5-py3-none-any.whl.metadata (3.7 kB)
  Using cached flatbuffers-25.9.23-py2.py3-none-any.whl.metadata (875 bytes)
  Using cached jh2-5.0.10-cp37-abi3-win_amd64.whl.metadata (4.1 kB)
  Using cached qh3-1.6.0-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
Using cached openmeteo_re

In [8]:
# ================================================================
# 01_DATA_COLLECTION.PY
# Anticipatory Solar Forecasting Research
# Collecting 5-year hourly meteorological data
# for 5 Saudi cities via Open-Meteo Historical API
# ================================================================

import openmeteo_requests
import requests_cache
import pandas as pd
import numpy as np
from retry_requests import retry
import os
import time

# ----------------------------------------------------------------
# 1. CITY COORDINATES
# ----------------------------------------------------------------
CITIES = {
    "Riyadh":  {"lat": 24.6877, "lon": 46.7219},
    "Jeddah":  {"lat": 21.5433, "lon": 39.1728},
    "Abha":    {"lat": 18.2164, "lon": 42.5053},
    "Dammam":  {"lat": 26.4207, "lon": 50.0888},
    "Qassim":  {"lat": 26.3260, "lon": 43.9750},
}

# ----------------------------------------------------------------
# 2. DATE RANGE
# ----------------------------------------------------------------
START_DATE = "2019-01-01"
END_DATE   = "2024-12-31"

# ----------------------------------------------------------------
# 3. VARIABLES TO COLLECT
# ----------------------------------------------------------------
HOURLY_VARIABLES = [
    "shortwave_radiation",      # GHI — our target variable (W/m²)
    "temperature_2m",           # Air temperature (°C)
    "relative_humidity_2m",     # Relative humidity (%)
    "cloud_cover",              # Total cloud cover (%)
    "wind_speed_10m",           # Wind speed (m/s) — dust proxy
    "wind_direction_10m",       # Wind direction (°) — dust proxy
    "surface_pressure",         # Atmospheric pressure (hPa)
    "precipitation",            # Precipitation (mm)
]

# ----------------------------------------------------------------
# 4. SETUP API SESSION (cache + auto-retry)
# Cache saves downloaded data locally so if script
# crashes you don't re-download everything from scratch
# ----------------------------------------------------------------
cache_session = requests_cache.CachedSession(
    '.cache', 
    expire_after=-1   # cache never expires
)
retry_session = retry(
    cache_session, 
    retries=5,           # retry up to 5 times if API fails
    backoff_factor=0.2   # wait a bit between retries
)
openmeteo = openmeteo_requests.Client(session=retry_session)

# ----------------------------------------------------------------
# 5. CREATE OUTPUT FOLDER
# ----------------------------------------------------------------
os.makedirs("data/raw", exist_ok=True)
print("✓ Output folder ready: data/raw/")

# ----------------------------------------------------------------
# 6. DATA COLLECTION FUNCTION
# ----------------------------------------------------------------
def collect_city_data(city_name, lat, lon):
    """
    Fetches 5-year hourly meteorological data
    for one city from Open-Meteo Historical API.
    Returns a clean pandas DataFrame.
    """
    print(f"\n{'='*55}")
    print(f"  Collecting: {city_name}")
    print(f"  Coordinates: ({lat}, {lon})")
    print(f"  Period: {START_DATE} to {END_DATE}")
    print(f"{'='*55}")

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude":        lat,
        "longitude":       lon,
        "start_date":      START_DATE,
        "end_date":        END_DATE,
        "hourly":          HOURLY_VARIABLES,
        "timezone":        "Asia/Riyadh",
        "wind_speed_unit": "ms",
    }

    # --- API Call ---
    responses = openmeteo.weather_api(url, params=params)
    response  = responses[0]

    print(f"  ✓ API response received")
    print(f"  ✓ Confirmed location: "
          f"{response.Latitude():.4f}°N, "
          f"{response.Longitude():.4f}°E")

    # --- Extract Hourly Data ---
    hourly = response.Hourly()

    # Build datetime index
    timestamps = pd.date_range(
        start = pd.Timestamp(hourly.Time(), unit="s", tz="Asia/Riyadh"),
        end   = pd.Timestamp(hourly.TimeEnd(), unit="s", tz="Asia/Riyadh"),
        freq  = pd.Timedelta(seconds=hourly.Interval()),
        inclusive = "left"
    )

    # Build dataframe
    df = pd.DataFrame({
        "timestamp":    timestamps,
        "GHI":          hourly.Variables(0).ValuesAsNumpy(),
        "temperature":  hourly.Variables(1).ValuesAsNumpy(),
        "humidity":     hourly.Variables(2).ValuesAsNumpy(),
        "cloud_cover":  hourly.Variables(3).ValuesAsNumpy(),
        "wind_speed":   hourly.Variables(4).ValuesAsNumpy(),
        "wind_dir":     hourly.Variables(5).ValuesAsNumpy(),
        "pressure":     hourly.Variables(6).ValuesAsNumpy(),
        "precipitation":hourly.Variables(7).ValuesAsNumpy(),
    })

    # --- Add city label ---
    df["city"] = city_name

    # --- Basic info ---
    print(f"  ✓ Rows collected: {len(df):,}")
    print(f"  ✓ Date range: {df['timestamp'].min()} "
          f"→ {df['timestamp'].max()}")
    print(f"  ✓ Missing GHI values: {df['GHI'].isna().sum()}")

    return df


# ----------------------------------------------------------------
# 7. COLLECT ALL CITIES
# ----------------------------------------------------------------
all_data = {}

for city_name, coords in CITIES.items():
    df = collect_city_data(
        city_name,
        coords["lat"],
        coords["lon"]
    )
    all_data[city_name] = df

    # Save individual city file
    output_path = f"data/raw/{city_name.lower()}_raw.csv"
    df.to_csv(output_path, index=False)
    print(f"  ✓ Saved: {output_path}")

    # Small pause between API calls — be polite to the server
    time.sleep(2)


# ----------------------------------------------------------------
# 8. QUICK VALIDATION — CHECK WHAT WE COLLECTED
# ----------------------------------------------------------------
print(f"\n{'='*55}")
print("  COLLECTION SUMMARY")
print(f"{'='*55}")

for city_name, df in all_data.items():
    print(f"\n  {city_name}:")
    print(f"    Total rows:      {len(df):,}")
    print(f"    Columns:         {list(df.columns)}")
    print(f"    GHI range:       "
          f"{df['GHI'].min():.1f} → {df['GHI'].max():.1f} W/m²")
    print(f"    Missing values:  {df.isna().sum().sum()}")
    print(f"    Years covered:   "
          f"{df['timestamp'].dt.year.unique().tolist()}")

print(f"\n{'='*55}")
print("  ✓ ALL CITIES COLLECTED SUCCESSFULLY")
print(f"{'='*55}")

✓ Output folder ready: data/raw/

  Collecting: Riyadh
  Coordinates: (24.6877, 46.7219)
  Period: 2019-01-01 to 2024-12-31
  ✓ API response received
  ✓ Confirmed location: 24.7100°N, 46.6881°E
  ✓ Rows collected: 52,608
  ✓ Date range: 2019-01-01 00:00:00+03:00 → 2024-12-31 23:00:00+03:00
  ✓ Missing GHI values: 0
  ✓ Saved: data/raw/riyadh_raw.csv

  Collecting: Jeddah
  Coordinates: (21.5433, 39.1728)
  Period: 2019-01-01 to 2024-12-31
  ✓ API response received
  ✓ Confirmed location: 21.5466°N, 39.2025°E
  ✓ Rows collected: 52,608
  ✓ Date range: 2019-01-01 00:00:00+03:00 → 2024-12-31 23:00:00+03:00
  ✓ Missing GHI values: 0
  ✓ Saved: data/raw/jeddah_raw.csv

  Collecting: Abha
  Coordinates: (18.2164, 42.5053)
  Period: 2019-01-01 to 2024-12-31
  ✓ API response received
  ✓ Confirmed location: 18.2425°N, 42.4976°E
  ✓ Rows collected: 52,608
  ✓ Date range: 2019-01-01 00:00:00+03:00 → 2024-12-31 23:00:00+03:00
  ✓ Missing GHI values: 0
  ✓ Saved: data/raw/abha_raw.csv

  Collecti